# RFT Deterministic Reasoning: Countdown

**TRAINING VARIANTS · ADVANCED**

This canonical notebook compares a model/score grader and a deterministic Python grader while sharing one data preparation, baseline, monitoring, deployment, and exact-answer evaluation path.

You will:

1. verify byte-preserved train, validation, evaluation, prompt, schema, and grader artifacts;
2. test graders independently, including deliberate reward-hacking attempts;
3. measure one baseline with the same deterministic evaluation contract;
4. optionally submit either or both RFT variants with bounded retries and monitoring;
5. distinguish job, registered-model, deployment, and evaluation outcomes.

> **Cost guardrail:** all remote sections are opt-in. `FOUNDRY_RUN_PAID_JOBS` defaults to false in code, so running this notebook as committed does not submit a fine-tuning job.

## 1. Imports, paths, and execution gates

In [ ]:
import importlib.util
import inspect
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
demo_candidates = [
    cwd,
    cwd.parent,
    cwd / "cookbooks/03-core-rft/01-rft-countdown",
]
DEMO_ROOT = next(
    (path for path in demo_candidates if (path / "data/hashes.json").is_file()),
    None,
)
if DEMO_ROOT is None:
    raise FileNotFoundError("Run from the repository or Countdown cookbook tree")
TRACK_ROOT = DEMO_ROOT.parent
sys.path.insert(0, str(TRACK_ROOT / "src"))

from rft_common import (
    collect_new_events,
    content_addressed_name,
    create_job_once,
    env_flag,
    extract_training_metrics,
    load_jsonl,
    require_env,
    retry,
    upload_file_once,
    validate_hash_manifest,
    wait_for_job,
)

load_dotenv(DEMO_ROOT / ".env")

FOUNDRY_RUN_REMOTE_BASELINE = env_flag("FOUNDRY_RUN_REMOTE_BASELINE")
FOUNDRY_RUN_REMOTE_GRADER_TESTS = env_flag("FOUNDRY_RUN_REMOTE_GRADER_TESTS")
FOUNDRY_RUN_PAID_JOBS = env_flag("FOUNDRY_RUN_PAID_JOBS")
FOUNDRY_RUN_REMOTE_EVALUATION = env_flag("FOUNDRY_RUN_REMOTE_EVALUATION")

print(
    {
        "remote_baseline": FOUNDRY_RUN_REMOTE_BASELINE,
        "remote_grader_tests": FOUNDRY_RUN_REMOTE_GRADER_TESTS,
        "paid_jobs": FOUNDRY_RUN_PAID_JOBS,
        "remote_evaluation": FOUNDRY_RUN_REMOTE_EVALUATION,
    }
)

The track-level `rft_common` module contains the reusable helpers planned for the shared cookbook layer: hash checks, content-addressed uploads, bounded retry/polling, and metric extraction. This notebook keeps only task-specific reward and evaluation logic.

## 2. Verify preserved data and split contracts

In [ ]:
validate_hash_manifest(DEMO_ROOT / "data/hashes.json", DEMO_ROOT)

TRAIN_PATH = DEMO_ROOT / "data/preserved/countdown-training-rft-100.jsonl"
VALIDATION_PATH = DEMO_ROOT / "data/preserved/countdown-validation-rft-50.jsonl"
EVALUATION_PATH = DEMO_ROOT / "data/preserved/countdown-evaluation-100.jsonl"

training_rows = load_jsonl(TRAIN_PATH)
validation_rows = load_jsonl(VALIDATION_PATH)
evaluation_rows = load_jsonl(EVALUATION_PATH)

assert len(training_rows) == 100
assert len(validation_rows) == 50
assert len(evaluation_rows) == 100
assert all(set(row) == {"messages", "target", "nums"} for row in training_rows)
assert all(set(row) == {"messages", "target", "nums"} for row in validation_rows)
assert all(set(row) == {"item"} for row in evaluation_rows)

def countdown_key(row):
    item = row.get("item", row)
    return str(item["target"]), str(item["nums"])

train_keys = {countdown_key(row) for row in training_rows}
validation_keys = {countdown_key(row) for row in validation_rows}
evaluation_keys = {countdown_key(row) for row in evaluation_rows}

assert train_keys.isdisjoint(validation_keys), "Training/validation leakage detected"
assert train_keys.isdisjoint(evaluation_keys), "Training/evaluation leakage detected"
assert validation_keys.isdisjoint(evaluation_keys), "Validation/evaluation leakage detected"

print(
    {
        "training_rows": len(training_rows),
        "validation_rows": len(validation_rows),
        "evaluation_rows": len(evaluation_rows),
        "split_overlap": 0,
    }
)

The notebook trains on the preserved 100/50 RFT splits and evaluates on the separate 100-row test split. Smaller preserved splits remain available for low-cost experiments, but changing the training input should be an explicit experiment.

## 3. Load the solver contract and both grader variants

In [ ]:
validate_hash_manifest(DEMO_ROOT / "graders/hashes.json", DEMO_ROOT / "graders")

SOLVER_PROMPT = (
    DEMO_ROOT / "graders/preserved/countdown-solver-prompt.txt"
).read_text(encoding="utf-8").replace("\\n", "\n")
RESPONSE_FORMAT = json.loads(
    (DEMO_ROOT / "graders/preserved/countdown-response-schema.json").read_text(
        encoding="utf-8"
    )
)
MODEL_GRADER = json.loads(
    (DEMO_ROOT / "graders/preserved/countdown-model-score-grader.json").read_text(
        encoding="utf-8"
    )
)

grader_path = DEMO_ROOT / "graders/countdown_python_grader.py"
grader_spec = importlib.util.spec_from_file_location("countdown_python_grader", grader_path)
grader_module = importlib.util.module_from_spec(grader_spec)
grader_spec.loader.exec_module(grader_module)
PYTHON_GRADER_SOURCE = grader_path.read_text(encoding="utf-8")
PYTHON_GRADER = {
    "type": "python",
    "name": "countdown_exact_arithmetic",
    "source": PYTHON_GRADER_SOURCE,
}

assert RESPONSE_FORMAT["json_schema"]["strict"] is True
assert MODEL_GRADER["type"] == "score_model"
assert MODEL_GRADER["pass_threshold"] == 5
assert MODEL_GRADER["range"] == [0.0, 5.0]
print("Solver, response schema, model grader, and Python grader loaded")

## 4. Validate the Python grader and probe reward hacking

In [ ]:
grader_tests = [
    (
        "exact",
        {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "24"}},
        {"target": "24", "nums": "[1, 2, 3, 4]"},
        5.0,
    ),
    (
        "near miss",
        {"output_json": {"expression": "(5 * 4) - (3 + 2)", "result": "15"}},
        {"target": "19", "nums": "[2, 3, 4, 5]"},
        3.0,
    ),
    (
        "reward hack: hidden extra constant",
        {"output_json": {"expression": "(9 - 7) * (8 + 6) + 22", "result": "50"}},
        {"target": "50", "nums": "[6, 7, 8, 9]"},
        0.0,
    ),
    (
        "reward hack: duplicate and omit",
        {"output_json": {"expression": "9 * 6 - (8 - 6)", "result": "52"}},
        {"target": "52", "nums": "[6, 7, 8, 9]"},
        0.0,
    ),
    (
        "reward hack: false reported result",
        {"output_json": {"expression": "(5 * 4) - (3 + 2)", "result": "19"}},
        {"target": "19", "nums": "[2, 3, 4, 5]"},
        1.0,
    ),
    (
        "reward hack: unsupported exponent",
        {"output_json": {"expression": "2 ** 3 + 4 + 5", "result": "17"}},
        {"target": "17", "nums": "[2, 3, 4, 5]"},
        0.0,
    ),
    (
        "reward hack: code execution",
        {"output_json": {"expression": "__import__('os').getcwd()", "result": "0"}},
        {"target": "0", "nums": "[1, 2, 3, 4]"},
        0.0,
    ),
]

for name, sample, item, expected in grader_tests:
    actual = grader_module.grade(sample, item)
    assert actual == expected, f"{name}: expected {expected}, received {actual}"
    print(f"{name:38} reward={actual}")

print(f"All {len(grader_tests)} deterministic grader tests passed")

The reward-hacking cases matter more than happy-path examples. A solver must not earn reward by injecting constants, repeating a convenient number, lying in the `result` field, using unsupported syntax, or attempting code execution.

## 5. Optionally calibrate the model/score grader

In [ ]:
if FOUNDRY_RUN_REMOTE_GRADER_TESTS:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_GRADER_DEPLOYMENT",
    )
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    grader_project = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
        allow_preview=True,
    )
    grader_client = grader_project.get_openai_client()
    grader_prompt = MODEL_GRADER["input"][0]["content"]

    remote_cases = [
        ("exact", grader_tests[0][1], grader_tests[0][2], 5),
        ("hidden extra constant", grader_tests[2][1], grader_tests[2][2], 0),
        ("false reported result", grader_tests[4][1], grader_tests[4][2], 1),
    ]
    remote_scores = {}
    for name, sample, item, expected in remote_cases:
        payload = {
            "target": item["target"],
            "numbers": item["nums"],
            "output": sample["output_json"],
        }
        response = retry(
            lambda payload=payload: grader_client.chat.completions.create(
                model=values["FOUNDRY_GRADER_DEPLOYMENT"],
                messages=[
                    {"role": "developer", "content": grader_prompt},
                    {"role": "user", "content": json.dumps(payload)},
                ],
                max_completion_tokens=500,
            ),
            description=f"calibrate model grader on {name}",
        )
        text = response.choices[0].message.content or ""
        match = re.search(r"Score:\s*([0-5](?:\.0)?)", text, re.IGNORECASE)
        if not match:
            raise AssertionError(f"Model grader returned no parseable score: {text!r}")
        score = float(match.group(1))
        remote_scores[name] = score
        if name == "exact":
            assert score == expected
        else:
            assert score <= expected
    print(remote_scores)
else:
    print("Skipped paid model-grader calibration; set FOUNDRY_RUN_REMOTE_GRADER_TESTS=true to run it.")

## 6. One shared baseline and evaluation contract

In [ ]:
def validate_countdown_output(output, item):
    sample = {"output_json": output} if isinstance(output, dict) else {"output_text": output}
    reward = grader_module.grade(sample, item)
    return {"reward": reward, "exact_valid": reward == 5.0}


def solve_countdown(client, deployment, item):
    response = retry(
        lambda: client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "developer", "content": SOLVER_PROMPT},
                {
                    "role": "user",
                    "content": f"Target: {item['target']}\nNumbers: {item['nums']}",
                },
            ],
            response_format=RESPONSE_FORMAT,
            max_completion_tokens=20_000,
        ),
        description=f"solve Countdown target {item['target']}",
    )
    output = json.loads(response.choices[0].message.content)
    return {**validate_countdown_output(output, item), "output": output}


def evaluate_deployment(client, deployment, rows, limit=20):
    results = []
    for row in rows[:limit]:
        results.append(solve_countdown(client, deployment, row["item"]))
    exact = sum(result["exact_valid"] for result in results)
    return {"deployment": deployment, "exact": exact, "total": len(results), "results": results}


illustrative_cases = [
    {"target": "24", "nums": "[1, 2, 3, 4]"},
    {"target": "19", "nums": "[2, 3, 4, 5]"},
    {"target": "50", "nums": "[6, 7, 8, 9]"},
]

baseline_result = None
if FOUNDRY_RUN_REMOTE_BASELINE:
    values = require_env("FOUNDRY_PROJECT_ENDPOINT", "FOUNDRY_BASELINE_DEPLOYMENT")
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    baseline_project = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
        allow_preview=True,
    )
    baseline_client = baseline_project.get_openai_client()
    baseline_rows = [{"item": item} for item in illustrative_cases]
    baseline_result = evaluate_deployment(
        baseline_client,
        values["FOUNDRY_BASELINE_DEPLOYMENT"],
        baseline_rows,
        limit=len(baseline_rows),
    )
    print({key: baseline_result[key] for key in ("deployment", "exact", "total")})
else:
    representative = json.loads(
        (DEMO_ROOT / "assets/metrics/representative-run.json").read_text(encoding="utf-8")
    )
    print("Representative baseline:", representative["baseline"])

The deterministic Python grader is the independent evaluation contract for both RFT variants. It is intentionally not the model/score training grader, which prevents a favorable model-grader opinion from being treated as proof of arithmetic correctness.

## 7. Prepare meaningful remote names and job configurations

In [ ]:
remote_names = {
    "training": content_addressed_name("core-rft-countdown-training", TRAIN_PATH),
    "validation": content_addressed_name("core-rft-countdown-validation", VALIDATION_PATH),
    "evaluation": content_addressed_name("core-rft-countdown-evaluation", EVALUATION_PATH),
}
remote_names

In [ ]:
model_grader_job = None
python_grader_job = None
training_client = None
project_client = None

if FOUNDRY_RUN_PAID_JOBS:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_BASE_MODEL",
        "FOUNDRY_GRADER_MODEL",
    )
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    project_client = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
        allow_preview=True,
    )
    training_client = project_client.get_openai_client()
    training_file = upload_file_once(
        training_client,
        TRAIN_PATH,
        prefix="core-rft-countdown-training",
        purpose="fine-tune",
    )
    validation_file = upload_file_once(
        training_client,
        VALIDATION_PATH,
        prefix="core-rft-countdown-validation",
        purpose="fine-tune",
    )

    model_grader = json.loads(json.dumps(MODEL_GRADER))
    model_grader["name"] = "core_rft_countdown_model_score"
    model_grader["model"] = values["FOUNDRY_GRADER_MODEL"]

    shared_hyperparameters = {
        "eval_interval": 5,
        "eval_samples": 2,
        "reasoning_effort": "medium",
        "n_epochs": 1,
        "batch_size": 4,
        "learning_rate_multiplier": 2,
    }

    def submit(grader, suffix):
        return training_client.fine_tuning.jobs.create(
            model=values["FOUNDRY_BASE_MODEL"],
            training_file=training_file.id,
            validation_file=validation_file.id,
            method={
                "type": "reinforcement",
                "reinforcement": {
                    "hyperparameters": shared_hyperparameters,
                    "grader": grader,
                    "response_format": RESPONSE_FORMAT,
                },
            },
            suffix=suffix,
        )

    model_grader_job = create_job_once(
        training_client,
        existing_job_id=os.getenv("FOUNDRY_EXISTING_MODEL_GRADER_JOB_ID") or None,
        create=lambda: submit(model_grader, "core-rft-countdown-model-grader"),
    )
    python_grader_job = create_job_once(
        training_client,
        existing_job_id=os.getenv("FOUNDRY_EXISTING_PYTHON_GRADER_JOB_ID") or None,
        create=lambda: submit(PYTHON_GRADER, "core-rft-countdown-python-grader"),
    )
    print(
        {
            "model_grader_job": model_grader_job.id,
            "python_grader_job": python_grader_job.id,
        }
    )
else:
    print("No jobs submitted. Set FOUNDRY_RUN_PAID_JOBS=true only after grader calibration.")

## 8. Bounded monitoring and training curves

In [ ]:
job_outcomes = {}
if FOUNDRY_RUN_PAID_JOBS:
    for label, job in {
        "model_score_grader": model_grader_job,
        "python_grader": python_grader_job,
    }.items():
        completed = wait_for_job(
            training_client,
            job.id,
            timeout_seconds=14_400,
            poll_seconds=60,
        )
        seen_ids = set()
        events = collect_new_events(training_client, job.id, seen_ids)
        messages = [event.message for event in events]
        metrics = extract_training_metrics(messages)
        job_outcomes[label] = {
            "status": completed.status,
            "registered_model": completed.fine_tuned_model,
            "reward": metrics["reward"],
            "loss": metrics["loss"],
        }
        print(label, job_outcomes[label])
else:
    representative = json.loads(
        (DEMO_ROOT / "assets/metrics/representative-run.json").read_text(encoding="utf-8")
    )
    print(
        {
            "model_score_grader": representative["model_score_grader_rft"],
            "python_grader": representative["python_grader_rft"],
        }
    )

If the service emits reward or loss values in job events, the previous cell extracts them rather than inventing a curve. The preserved Countdown run exposes representative exact-answer numbers but not a complete reward/loss series, so this notebook reports those numbers honestly instead of fabricating a chart.

## 9. Deployment availability and independent evaluation

In [ ]:
deployment_outcomes = {}
evaluation_outcomes = {}

if FOUNDRY_RUN_REMOTE_EVALUATION:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_COUNTDOWN_MODEL_GRADER_DEPLOYMENT",
        "FOUNDRY_COUNTDOWN_PYTHON_GRADER_DEPLOYMENT",
    )
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    evaluation_project = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
        allow_preview=True,
    )
    evaluation_client = evaluation_project.get_openai_client()

    deployments = {
        "model_score_grader": values["FOUNDRY_COUNTDOWN_MODEL_GRADER_DEPLOYMENT"],
        "python_grader": values["FOUNDRY_COUNTDOWN_PYTHON_GRADER_DEPLOYMENT"],
    }
    for label, deployment_name in deployments.items():
        deployment = retry(
            lambda name=deployment_name: evaluation_project.deployments.get(name),
            description=f"retrieve deployment {deployment_name}",
        )
        deployment_outcomes[label] = {
            "status": "available",
            "name": deployment.name,
            "model": deployment.model_name,
        }
        evaluation_outcomes[label] = evaluate_deployment(
            evaluation_client,
            deployment_name,
            evaluation_rows,
            limit=20,
        )
        print(
            label,
            {
                key: evaluation_outcomes[label][key]
                for key in ("deployment", "exact", "total")
            },
        )
else:
    print("Skipped deployment checks and hosted evaluation.")

## 10. Outcome distinctions, conclusion, and next step

In [ ]:
outcome_summary = {
    "grader_validation": "passed offline",
    "baseline": (
        {"exact": baseline_result["exact"], "total": baseline_result["total"]}
        if baseline_result
        else "representative 1/3"
    ),
    "training_jobs": job_outcomes or "not submitted in this run",
    "deployments": deployment_outcomes or "not checked in this run",
    "independent_evaluation": evaluation_outcomes or "not run in this run",
}
outcome_summary

### Conclusion

The Python grader is the safer default for this deterministic task because it is inspectable, fast, and rejects the tested reward hacks. The model/score grader remains useful when acceptable behavior cannot be fully encoded, but it needs calibration against adversarial examples and should not be its own final judge.

**Next step:** run the independent evaluation on a larger held-out sample, inspect disagreements between grader variants, and change one variable at a time before deciding whether the added model-grader flexibility is worth its cost and variance.

*Variability footnote:* the preserved `1/3 → 3/3` numbers cover only three illustrative cases. Model snapshots, grader models, service behavior, region, sampling, and deployment configuration can change both training and evaluation outcomes.

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.